# Speech to text with `indic-transcribe`

Transcribe short audio clips in 25 Indian languages plus English.

| | |
|---|---|
| Endpoint | `POST /v1/audio/transcriptions` (multipart, OpenAI-compatible) |
| Model | `indic-transcribe` |
| Input | audio file, up to ~30 seconds per request |
| Output | `{"text": "..."}` |
| Billing | per second of audio |

**You need** a Bodhan API key from [console.bodhan.ai](https://console.bodhan.ai), exported as `BODHAN_API_KEY`. Full reference: [console.bodhan.ai/api-docs](https://console.bodhan.ai/api-docs/).

In [ ]:
%pip install -q requests==2.32.3

In [ ]:
import os, json, requests

BASE_URL = os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai")
API_KEY = os.environ["BODHAN_API_KEY"]  # export BODHAN_API_KEY=... before starting Jupyter
HEADERS = {"Authorization": f"Bearer {API_KEY}"}


def raise_for_bodhan(resp):
    """Bodhan errors are JSON: {"error": {"message", "code", "request_id"}}. Surface them readably."""
    if resp.ok:
        return resp
    try:
        err = resp.json()["error"]
        raise RuntimeError(f"{resp.status_code} {err.get('code')}: {err.get('message')} (request_id={err.get('request_id')})")
    except (ValueError, KeyError):
        resp.raise_for_status()

## 1. Transcribe a file

The repo ships a short Hindi clip in `sample_data/`. Pass `language` when you know it; the model uses it as a strong hint.

In [ ]:
AUDIO = "../../sample_data/sample_hindi.wav"

with open(AUDIO, "rb") as fh:
    resp = requests.post(
        f"{BASE_URL}/v1/audio/transcriptions",
        headers=HEADERS,
        files={"file": fh},
        data={"model": "indic-transcribe", "language": "hi"},
        timeout=60,
    )
raise_for_bodhan(resp)
print(resp.json()["text"])

## 2. Wrap it in a function

Everything else in this notebook builds on this helper.

In [ ]:
def transcribe(path: str, language: str | None = None) -> str:
    data = {"model": "indic-transcribe"}
    if language:
        data["language"] = language
    with open(path, "rb") as fh:
        resp = requests.post(f"{BASE_URL}/v1/audio/transcriptions", headers=HEADERS, files={"file": fh}, data=data, timeout=60)
    return raise_for_bodhan(resp).json()["text"]


transcribe(AUDIO, "hi")

## 3. Supported languages

ISO codes, not BCP-47 (`hi`, not `hi-IN`). Omit `language` and the model will detect it, at some cost to accuracy on short clips.

In [ ]:
LANGUAGES = {
    "as": "Assamese", "bhb": "Bhili", "bho": "Bhojpuri", "bn": "Bengali", "brx": "Bodo", "doi": "Dogri",
    "en": "English", "gu": "Gujarati", "hi": "Hindi", "kn": "Kannada", "kok": "Konkani", "ks": "Kashmiri",
    "mai": "Maithili", "ml": "Malayalam", "mni": "Manipuri", "mr": "Marathi", "ne": "Nepali", "or": "Odia",
    "pa": "Punjabi", "sa": "Sanskrit", "sat": "Santali", "sd": "Sindhi", "ta": "Tamil", "te": "Telugu", "ur": "Urdu",
}
print(len(LANGUAGES), "languages:", ", ".join(LANGUAGES))

## 4. Longer audio: chunk it

Each request takes about 30 seconds of audio. For a lecture or a call recording, split on silence (or fixed windows with a little overlap) and concatenate the transcripts. This example uses fixed 25-second windows with the standard library only; for silence-aware splitting look at `pydub` or `webrtcvad`.

In [ ]:
import wave, io, contextlib


def chunk_wav(path: str, seconds: float = 25.0):
    """Yield (index, bytes) for fixed-length WAV windows. Works for PCM WAV files."""
    with contextlib.closing(wave.open(path, "rb")) as wf:
        params = wf.getparams()
        frames_per_chunk = int(params.framerate * seconds)
        idx = 0
        while True:
            frames = wf.readframes(frames_per_chunk)
            if not frames:
                break
            buf = io.BytesIO()
            with contextlib.closing(wave.open(buf, "wb")) as out:
                out.setparams(params)
                out.writeframes(frames)
            yield idx, buf.getvalue()
            idx += 1


def transcribe_long(path: str, language: str | None = None) -> str:
    parts = []
    for idx, blob in chunk_wav(path):
        data = {"model": "indic-transcribe", **({"language": language} if language else {})}
        resp = requests.post(f"{BASE_URL}/v1/audio/transcriptions", headers=HEADERS,
                             files={"file": (f"chunk{idx}.wav", blob, "audio/wav")}, data=data, timeout=60)
        parts.append(raise_for_bodhan(resp).json()["text"])
    return " ".join(parts)


transcribe_long(AUDIO, "hi")  # the sample is short, so this is a single chunk

## 5. Errors and limits

- `401` — bad or missing key. `429` — rate limit or monthly budget reached; back off and retry after the window.
- Look at `x-ratelimit-limit-requests` and `x-ratelimit-limit-parallel-requests` on any response to see your key's limits.
- The OpenAI Python SDK also works: `client.audio.transcriptions.create(model="indic-transcribe", file=open(path, "rb"))` with `base_url="https://api.bodhan.ai/v1"`. See [`integrations/openai_sdk/`](../../integrations/openai_sdk/).

**Next:** feed the transcript into [`translate`](../translate/translate.ipynb) or read it back with [`text-to-speech`](../text-to-speech/text_to_speech.ipynb).